# Convert the OPRD-100.json dataset to ORD format

## Loading packages and data

In [150]:
import json
from ord_schema import message_helpers
from ord_schema import validations
from ord_schema.logging import get_logger
from ord_schema.proto import dataset_pb2
from ord_schema.proto import reaction_pb2
from ord_schema import updates
from datetime import datetime
import pandas as pd

In [194]:
pd.set_option('display.max_rows', None)

In [4]:
# read in the json data file
filename = "../OPRD-100.json"
with open(filename, 'r', encoding= 'utf-8') as file:
    data = json.load(file)

In [7]:
# check the first reaction and consider the overall structure
data[0]

{'Reaction': 'CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(=O)N2CCCC[C@H]2C(=O)C[C@H]([C@H](C)C[C@@H]2CC[C@@H](O)[C@H](OC)C2)CC(=O)[C@H](C)/C=C(\\C)[C@@H](O)[C@@H](OC)C(=O)[C@H](C)C[C@H](C)/C=C/C=C/C=C/1C.CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c1ccccc1>>CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(=O)N2CCCC[C@H]2C(=O)C[C@H]([C@H](C)C[C@@H]2CC[C@@H](OCCO[Si](c3ccccc3)(c3ccccc3)C(C)(C)C)[C@H](OC)C2)CC(=O)[C@H](C)/C=C(\\C)[C@@H](O)[C@@H](OC)C(=O)[C@H](C)C[C@H](C)/C=C/C=C/C=C/1C',
 'Reference': 'acs.oprd.1c00456',
 'Location': {'Type': 'Table AND Scheme AND Scheme', 'Num': '4; 2; 5'},
 'Full text of reaction': None,
 'Amounts': 'CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(=O)N2CCCC[C@H]2C(=O)C[C@H]([C@H](C)C[C@@H]2CC[C@@H](O)[C@H](OC)C2)CC(=O)[C@H](C)/C=C(\\C)[C@@H](O)[C@@H](OC)C(=O)[C@H](C)C[C@H](C)/C=C/C=C/C=C/1C (3 g), CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c1ccccc1 (2.5 equiv)',
 'Steps': [{'Step': 1,
   'Yield': {'Isolated': None,
    'Flow': None,
    'Conversion wrt 

Things to consider:
- JSON data includes reactions with 2 or more steps. How to translate these into ORD? Need to review some of the source data to determine best route forward.
- Amounts of reactants includes values in equiv format. This is not compatible with ORD so will need to calculate the moles/mass.
- Lots of options for yield. Will need to look at these in more detail to determine correct translations.

## Split the dataset into single and multi-step reactions

Multi step reactions will require more consideration so we need a subset of them for investigation.

In [11]:
# Create some empty lists for sorting the reactions into
one_step = []
two_step = []
more_step = []
unknown_step = []

In [12]:
# Iterate through the reactions and add them to the appropriate lists
for rxn in data:
    steps = len(rxn['Steps'])
    if steps == 1:
        one_step.append(rxn)
    elif steps == 2:
        two_step.append(rxn)
    elif steps > 2:
        more_step.append(rxn)
    else:
        unknown_step.append(rxn)

In [17]:
# Count the lists
f"""{len(one_step)} single step reactions, {len(two_step)} two step reactions, {len(more_step)} reactions with more than two steps, and {len(unknown_step)} reactions which couldn't be parsed."""

"3288 single step reactions, 363 two step reactions, 227 reactions with more than two steps, and 0 reactions which couldn't be parsed."

In [20]:
# Check that the first reaction was translated correctly
one_step[0] == data[0]

True

In [32]:
# Write the step subsets to separate json files

with open("one_step.json", "w", encoding='utf-8') as f:
    json.dump(one_step, f, indent= 4)

with open("two_step.json", "w", encoding='utf-8') as f:
    json.dump(two_step, f, indent= 4)

with open("more_step.json", "w", encoding='utf-8') as f:
    json.dump(more_step, f, indent= 4)

with open("unknown_step.json", "w", encoding='utf-8') as f:
    json.dump(unknown_step, f, indent= 4)

## Functions

In [123]:
# check if json field has data and if so, create the corresponding ord field

def field_check(input, output):
    """check if json input field has data and if so, create the corresponding ord output field"""
    if input is not None:
        output
    else:
        pass

In [143]:
def parse_amounts(amounts):
    """Interpret the json amounts field, split it into reactants, and their corresponding amounts."""
    reactant_list = str(amounts).split("), ")
    print(reactant_list)
    for chem in reactant_list:
        chem_parts = chem.split(" (")
        chem_parts[-1] = chem_parts[-1].replace(")", "")
        print(chem_parts)

        # to do: need some regex to parse the amounts field and extract mass, moles, and equiv into a dictionary with smiles. This can then be used to calculate additional measurements from equiv. 
        
        #ord_chem = ord_rxn.inputs['single'].components.add()        
        #ord_chem.CopyFrom(
        #    message_helpers.build_compound(
        #        smiles= chem_parts[0],
        #        role= 'reactant',
        #        #amount= chem_parts[1]
        #    )
        #)

In [144]:
# testing parse_amounts
amounts = one_step[99]['Amounts']

parse_amounts(amounts)



['O=C(O)[C@H]([C@H](O)c1ccc2c(c1)OCCO2)N(Cc1ccccc1)Cc1ccccc1 (1 equiv', 'C1CCNC1 (2 equiv)']
['O=C(O)[C@H]([C@H](O)c1ccc2c(c1)OCCO2)N(Cc1ccccc1)Cc1ccccc1', '1 equiv']
['C1CCNC1', '2 equiv']


In [147]:
# extract the amounts for every single step reaction. Use this list to guide development of the parser
reactant_1_amounts = []
reactant_2_amounts = []

for rxn in one_step:
    reactant_list = str(amounts).split("), ")
    chem_parts_1 = reactant_list[0].split(" (")
    chem_parts_2 = reactant_list[1].split(" (")
    chem_parts_1[-1] = chem_parts_1[-1].replace(")", "")
    chem_parts_2[-1] = chem_parts_2[-1].replace(")", "")
    reactant_1_amounts.append(chem_parts_1[-1])
    reactant_2_amounts.append(chem_parts_2[-1])
    

In [175]:
# create a DataFrame from the Amounts in the json single step data
amounts = []
for rxn in one_step:
    amounts.append(rxn['Amounts'])

amounts_df = pd.DataFrame(amounts, columns=['Amounts'])

In [176]:
# inspect the DataFrame
print(len(amounts_df))
print(amounts_df.info())
print(amounts_df.head())

3288
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3288 entries, 0 to 3287
Data columns (total 1 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Amounts  1815 non-null   object
dtypes: object(1)
memory usage: 25.8+ KB
None
                                             Amounts
0  CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...
1  CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...
2  CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...
3  CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...
4  CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...


In [177]:
# Split the Amounts column into two separate reactants
amounts_df[['reactant_1', 'reactant_2']] = amounts_df['Amounts'].str.split('\), ', expand=True)

In [178]:
# split each reactant into smiles and amounts
amounts_df[['reactant_1_smiles', 'reactant_1_amount']] = amounts_df['reactant_1'].str.split(' \(', n=1, expand= True)
amounts_df[['reactant_2_smiles', 'reactant_2_amount']] = amounts_df['reactant_2'].str.split(' \(', n=1, expand= True)

In [184]:
# strip the trailing ')' from amounts
amounts_df['reactant_1_amount'] = amounts_df['reactant_1_amount'].str.rstrip(')')
amounts_df['reactant_2_amount'] = amounts_df['reactant_2_amount'].str.rstrip(')')

In [197]:
# inspect how many rows are missing amounts
amounts_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3288 entries, 0 to 3287
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Amounts            1815 non-null   object
 1   reactant_1         1815 non-null   object
 2   reactant_2         650 non-null    object
 3   reactant_1_smiles  1815 non-null   object
 4   reactant_1_amount  1814 non-null   object
 5   reactant_2_smiles  650 non-null    object
 6   reactant_2_amount  650 non-null    object
dtypes: object(7)
memory usage: 179.9+ KB


In [196]:
amounts_df.head(100)

,Amounts,reactant_1,reactant_2,reactant_1_smiles,reactant_1_amount,reactant_2_smiles,reactant_2_amount
0,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,3 g,CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c...,2.5 equiv
1,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,3 g,CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c...,2.5 equiv
2,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,3 g,CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c...,2.5 equiv
3,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,3 g,CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c...,2.5 equiv
4,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,3 g,CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c...,2.5 equiv
5,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,3 g,CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c...,2.5 equiv
6,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,3 g,CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c...,2.5 equiv
7,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,3 g,CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c...,2.5 equiv
8,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,3 g,CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c...,2 equiv
9,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c...,CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(...,3 g,CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c...,2 equiv


In [195]:
# inspect the range of options in reactant 1 amounts. Expect these values to be somewhat consistent within papers/tables.
amounts_df['reactant_1_amount'].value_counts()

reactant_1_amount
1 equiv                                                                                 177
1.0 mmol                                                                                112
1.0 equiv                                                                                98
30.0 mmol, 1 equiv                                                                       89
0.1 M                                                                                    81
1.2 equiv                                                                                59
0.35 mmol                                                                                45
126 mg, 0.503 mmol, 1.5 equiv                                                            43
1.5 equiv                                                                                32
1 equiv, 8 portions                                                                      30
2 mmol                                                        

Note that ORD reaction records require amounts for the reactants, and only accept mass, mole, or volume types. From this count of reactant_1 it looks like there would be circa 500 reactions in the single step set which would qualify.

In [ ]:
#amounts_df['reactant_1_eq'] = amounts_df['reactant_1_amount'].str.extract(r"\d", expand=False)

## Convert the single step reactions

In [113]:
# Select a slice of the reactions for testing 
subset = one_step[:100]

In [137]:

ord_one_step = []

for rxn in subset:
    ord_rxn = reaction_pb2.Reaction()
    if rxn['Reaction'] is not None:
        ord_rxn.identifiers.add(type= "REACTION_SMILES", value= rxn['Reaction'])
    else:
        pass
    if rxn['Reference'] is not None:
        ord_rxn.provenance.doi = f"dx.doi.org/10.1021/{rxn['Reference']}"
    else:
        pass
    if rxn['Location'] is not None:
        ord_rxn.provenance.reaction_metadata['Location_Type'].CopyFrom(reaction_pb2.Data(string_value = rxn['Location']['Type'], description = 'Location types where reaction is reported in publication'))
    else:
        pass
    if rxn['Location'] is not None:
        ord_rxn.provenance.reaction_metadata['Location_Num'].CopyFrom(reaction_pb2.Data(string_value = rxn['Location']['Num'], description = 'Scheme, figure or table numbers where reaction is reported in publication'))
    else:
        pass
    ord_rxn.provenance.record_created.time.value = datetime.now().strftime("%Y/%m/%d, %H:%M:%S")
    ord_rxn.provenance.record_created.person.CopyFrom(
        reaction_pb2.Person(name= "Benjamin J. Deadman", organization= "ORD", orcid= "0000-0001-8463-8199", email= "ben@bjdeadman.co.uk"))

    if rxn['Full text of reaction'] is not None:
        ord_rxn.notes.procedure_details = rxn['Full text of reaction']
    else:
        pass

    if rxn['Amounts'] is not None:
        reactant_list = str(rxn['Amounts']).split("), ")
        #print(reactant_list)
        for chem in reactant_list:
            chem_parts = chem.split(" (")
            chem_parts[-1] = chem_parts[-1].replace(")", "")
            #print(chem_parts)
            ord_chem = ord_rxn.inputs['single'].components.add()
            ord_chem.CopyFrom(
                message_helpers.build_compound(
                    smiles= chem_parts[0],
                    role= 'reactant',
                    #amount= chem_parts[1]
                )
            
            )
        


    ord_one_step.append(ord_rxn)
    

In [138]:
ord_one_step

[identifiers {
   type: REACTION_SMILES
   value: "CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(=O)N2CCCC[C@H]2C(=O)C[C@H]([C@H](C)C[C@@H]2CC[C@@H](O)[C@H](OC)C2)CC(=O)[C@H](C)/C=C(\\C)[C@@H](O)[C@@H](OC)C(=O)[C@H](C)C[C@H](C)/C=C/C=C/C=C/1C.CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c1ccccc1)c1ccccc1>>CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(=O)N2CCCC[C@H]2C(=O)C[C@H]([C@H](C)C[C@@H]2CC[C@@H](OCCO[Si](c3ccccc3)(c3ccccc3)C(C)(C)C)[C@H](OC)C2)CC(=O)[C@H](C)/C=C(\\C)[C@@H](O)[C@@H](OC)C(=O)[C@H](C)C[C@H](C)/C=C/C=C/C=C/1C"
 }
 inputs {
   key: "single"
   value {
     components {
       identifiers {
         type: SMILES
         value: "CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(=O)N2CCCC[C@H]2C(=O)C[C@H]([C@H](C)C[C@@H]2CC[C@@H](O)[C@H](OC)C2)CC(=O)[C@H](C)/C=C(\\C)[C@@H](O)[C@@H](OC)C(=O)[C@H](C)C[C@H](C)/C=C/C=C/C=C/1C"
       }
       reaction_role: REACTANT
     }
     components {
       identifiers {
         type: SMILES
         value: "CC(C)(C)[Si](OCCOS(=O)(=O)C(F)(F)F)(c